# URLBERT Baseline (MODEL 1)

In [ ]:
%pip install -q "transformers>=4.40,<5" tensorboard tqdm

In [ ]:
import sys
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torch.utils.tensorboard import SummaryWriter
from transformers import BertConfig, BertForMaskedLM, BertTokenizerFast
from tqdm.auto import tqdm

sys.path.append("../src")
from load_data import load_splits
from evaluate import evaluate, regex_baseline, save_preds

# Setup
- `SMOKE = True` runs a quick check on 2000 rows before the full run

In [ ]:
SMOKE = True

SEED = 42
BATCH_SIZE = 64
LR = 2e-5
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 1 if SMOKE else 5
EVAL_EVERY = 10 if SMOKE else 500  # steps between validations
PATIENCE = 3                        # evals without improvement before stopping
MIN_DELTA = 1e-4

TAG = "urlbert_smoke" if SMOKE else "urlbert"
URLBERT_DIR = "../models/urlbert"
RESULTS = "../results"
CKPT = f"../checkpoints/{TAG}_model1.pt"
LOG = f"{RESULTS}/{TAG}_train_log.csv"
os.makedirs(RESULTS, exist_ok=True)
os.makedirs("../checkpoints", exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
use_amp = device == "cuda"
print("device:", device, "| torch", torch.__version__)

# Load Data

In [ ]:
splits = load_splits()
if SMOKE:
    splits = {s: d.sample(2000 if s == "train" else 500, random_state=SEED) for s, d in splits.items()}
print({s: len(d) for s, d in splits.items()})

# Preprocess
- URLBERT's WordPiece tokenizer (vocab trained on URLs)

In [ ]:
tokenizer = BertTokenizerFast(vocab_file=f"{URLBERT_DIR}/bert_tokenizer/vocab.txt", do_lower_case=True)
for u in splits["train"].URL[:3]:
    print(u, "->", tokenizer.tokenize(u))

In [ ]:
#choose max token length (99th percentile)
enc = tokenizer(splits["train"].URL.tolist())["input_ids"]
lengths = np.array([len(x) for x in enc])
unk_rate = np.mean(np.concatenate(enc) == tokenizer.unk_token_id)
MAX_LEN = min(int(np.percentile(lengths, 99)), 128)

print(f"token length: median {np.median(lengths):.0f}, p99 {np.percentile(lengths, 99):.0f}, max {lengths.max()}")
print(f"MAX_LEN = {MAX_LEN}, truncated = {(lengths > MAX_LEN).mean():.2%}, UNK rate = {unk_rate:.4%}")
assert unk_rate < 0.01, "tokenizer is not loading the vocab properly"

In [ ]:
class URLDataset(Dataset):
    def __init__(self, df):
        self.ids = tokenizer(df.URL.tolist(), truncation=True, max_length=MAX_LEN)["input_ids"]
        self.labels = df.label.tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return torch.tensor(self.ids[i]), self.labels[i]


def collate(batch):
    # pad to longest in batch
    ids, labels = zip(*batch)
    ids = pad_sequence(ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    return ids, (ids != tokenizer.pad_token_id).long(), torch.tensor(labels)


datasets = {s: URLDataset(d) for s, d in splits.items()}
train_loader = DataLoader(datasets["train"], batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
val_loader = DataLoader(datasets["val"], batch_size=256, collate_fn=collate)
test_loader = DataLoader(datasets["test"], batch_size=256, collate_fn=collate)

# Model
- pretrained URLBERT encoder + linear layer on [CLS] (same head as URLBERT's phishing fine-tune)

In [ ]:
config = BertConfig.from_json_file(f"{URLBERT_DIR}/bert_config/config.json")
config.vocab_size = 5000

mlm = BertForMaskedLM(config)
state = torch.load(f"{URLBERT_DIR}/urlBERT.pt", map_location="cpu", weights_only=True)
state.pop("bert.embeddings.position_ids", None)  # older checkpoints store this buffer
mlm.load_state_dict(state)  # strict: errors if weights don't match


class URLBERTClassifier(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(768, 2)

    def forward(self, input_ids, attention_mask):
        h = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        return self.fc(self.dropout(h))


model = URLBERTClassifier(mlm.bert).to(device)
del mlm, state
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

# Training
- AdamW, linear warmup (6% of steps) then linear decay
- validate every `EVAL_EVERY` steps, early stopping on val loss, keep best checkpoint
- live curves: `tensorboard --logdir runs` (from repo root)

In [ ]:
loss_fn = nn.CrossEntropyLoss()


@torch.no_grad()
def run_eval(loader):
    # returns mean loss and p_phish (label 0)
    model.eval()
    total, probs = 0.0, []
    for ids, mask, y in loader:
        ids, mask, y = ids.to(device), mask.to(device), y.to(device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            logits = model(ids, mask).float()
        total += loss_fn(logits, y).item() * len(y)
        probs.append(torch.softmax(logits, -1)[:, 0].cpu())
    return total / len(loader.dataset), torch.cat(probs).numpy()


def predict(urls):
    ds = URLDataset(pd.DataFrame({"URL": list(urls), "label": 1}))
    return run_eval(DataLoader(ds, batch_size=256, collate_fn=collate))[1]

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = MAX_EPOCHS * len(train_loader)
warmup = int(0.06 * total_steps)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lambda s: s / max(1, warmup) if s < warmup else max(0.0, (total_steps - s) / max(1, total_steps - warmup))
)
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
writer = SummaryWriter(f"../runs/{TAG}")

log, running = [], []
best_val, bad, step, stopped = float("inf"), 0, 0, False

for epoch in range(1, MAX_EPOCHS + 1):
    for ids, mask, y in tqdm(train_loader, desc=f"epoch {epoch}"):
        model.train()
        ids, mask, y = ids.to(device), mask.to(device), y.to(device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            loss = loss_fn(model(ids, mask), y)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        step += 1
        running.append(loss.item())
        writer.add_scalar("loss/train_step", running[-1], step)

        if step % EVAL_EVERY == 0:
            val_loss, p_val = run_eval(val_loader)
            m = evaluate(datasets["val"].labels, p_val)
            improved = val_loss < best_val - MIN_DELTA
            if improved:
                best_val, bad = val_loss, 0
                torch.save(model.state_dict(), CKPT)
            else:
                bad += 1

            row = {"step": step, "epoch": epoch, "train_loss": np.mean(running), "val_loss": val_loss,
                   "val_acc": m["accuracy"], "val_recall": m["recall"], "val_f1": m["f1"],
                   "lr": scheduler.get_last_lr()[0], "best": improved}
            log.append(row)
            running = []
            pd.DataFrame(log).to_csv(LOG, index=False)

            writer.add_scalars("loss", {"train": row["train_loss"], "val": val_loss}, step)
            writer.add_scalar("val/f1", m["f1"], step)
            writer.add_scalar("val/recall", m["recall"], step)
            writer.add_scalar("lr", row["lr"], step)
            print(f"step {step} | train {row['train_loss']:.4f} | val {val_loss:.4f} | f1 {m['f1']:.4f}" + (" | best" if improved else ""))

            if bad >= PATIENCE:
                print(f"early stopping: val loss did not improve for {PATIENCE} evals")
                stopped = True
                break
    if stopped:
        break

writer.close()

## Loss Curve

In [ ]:
log_df = pd.read_csv(LOG)
best = log_df.loc[log_df.val_loss.idxmin()]

plt.plot(log_df.step, log_df.train_loss, label="train")
plt.plot(log_df.step, log_df.val_loss, label="validation")
plt.axvline(best.step, ls="--", c="grey", label=f"best checkpoint (step {int(best.step)})")
if stopped:
    plt.axvline(log_df.step.iloc[-1], ls=":", c="red", label="early stop")
plt.xlabel("Step")
plt.ylabel("Cross-entropy loss")
plt.title("URLBERT Training and Validation Loss")
plt.legend()
plt.savefig(f"{RESULTS}/{TAG}_loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()

display(log_df.tail())

# Evaluate on Test (test_original)
- best checkpoint, compared against the regex reference rule

In [ ]:
model.load_state_dict(torch.load(CKPT, map_location=device, weights_only=True))
test_loss, p_test = run_eval(test_loader)
test = splits["test"]

test_metrics = evaluate(test.label, p_test)
save_preds(f"{RESULTS}/{TAG}_test_original.csv", test.URL, test.label, p_test)
with open(f"{RESULTS}/{TAG}_test_original.json", "w") as f:
    json.dump(test_metrics, f, indent=1)

comparison = pd.DataFrame({
    "URLBERT": test_metrics,
    "regex rule": evaluate(test.label, regex_baseline(test.URL)),
}).drop("confusion_matrix").T
display(comparison.astype(float).round(4))

In [ ]:
sns.heatmap(test_metrics["confusion_matrix"], annot=True, fmt="d", cmap="Blues",
            xticklabels=["Phishing", "Legitimate"], yticklabels=["Phishing", "Legitimate"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("URLBERT Confusion Matrix (test_original)")
plt.show()

# Adversarial Evaluation
TODO once test_adversarial is ready: `p_adv = predict(adv.URL)`, then `adversarial_metrics(...)` from `evaluate.py`